# 00b – Keras Spike: Keras 3 with PyTorch backend on Apple Silicon

## 1. Goal
Short experiment (timebox 1 h) before notebook 05: does **Keras 3 with the PyTorch backend** run reliably on our Macs (MPS)?
At the end we decide: Keras or plain PyTorch for notebooks 05–08.

Steps:
1. Install Keras and set the backend
2. Check which device is used
3. Load EfficientNetB0 and look at it
4. Pass 64 of our own crops through the network
5. Short training with a frozen base network
6. Measure time: MPS vs. CPU
7. Test gradients for Grad-CAM
8. Write down the decision

## 2. Analysis
### Step 1 – Install Keras and set the backend
Keras itself does not compute anything, it needs a **backend**. The backend is chosen with the environment variable `KERAS_BACKEND`,
and it is read **once, at the first `import keras`**. So the variable must be set before Keras is imported.

In [1]:
%pip install keras

  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 10.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.7/137.7 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 18.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 737.4/737.4 kB 15.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 408.2/408.2 kB 9.2 MB/s eta 0:00:00a 0:00:01
Using cached rich-15.0.0-py3-none-any.whl (310 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.0/97.0 kB 12.8 MB/s eta 0:00:00
Using cached mdurl-0.1.2-py3-none-any.whl (10.0 kB)

[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
os.environ["KERAS_BACKEND"] = "torch"   # must come BEFORE the first import of keras

import keras
import torch

print("Keras version:  ", keras.__version__)
print("Keras backend:  ", keras.backend.backend())
print("PyTorch version:", torch.__version__)

Keras version:   3.15.1
Keras backend:   torch
PyTorch version: 2.15.0


**Observation:**

important: set the backend first, before importing keras, we have to define its backend torch. otherwise it wants to go to its standard tensorflow, which ist not installed in the venv at the moment. 

### Step 2 – Check which device is used
With the PyTorch backend, every Keras tensor is in fact a PyTorch tensor, and every PyTorch tensor lives on a **device**:
`cpu`, `cuda` (NVIDIA GPU) or `mps` (Apple GPU). Keras picks a default device on its own.
With `keras.device(...)` we can force a device for a block of code. We will need this in step 6 to compare MPS and CPU.

In [3]:
print("MPS available:", torch.backends.mps.is_available())

# 1) a tensor created by Keras
x = keras.ops.ones((2, 3))
print("Type of x:    ", type(x))
print("x is on:      ", x.device)

# 2) the weights of a Keras layer
layer = keras.layers.Dense(4)
layer.build((None, 3))      # creates the weights for an input with 3 features
print("Weights on:   ", layer.kernel.value.device)

# 3) force the CPU for one block
with keras.device("cpu"):
    y = keras.ops.ones((2, 3))
print("y is on:      ", y.device)

MPS available: True
Type of x:     <class 'torch.Tensor'>
x is on:       mps:0
Weights on:    mps:0
y is on:       cpu


**Observation:** 
Keras uses MPS by default; tensors and weights are on the GPU. With keras.device("cpu") we can force the CPU.